In [1]:
import pandas as pd
import os
# Ensure the raw data directory exists
os.makedirs('../data/raw', exist_ok=True)

# football-data.co.uk uses a specific URL format: mmz4281/{season}/E0.csv
# E0 stands for English Premier League. Season 2526 represents 2025/2026.
season_str = "2526"
url = f"https://www.football-data.co.uk/mmz4281/{season_str}/E0.csv"

print(f"Fetching match results from {url}...")
df_results = pd.read_csv(url)

# Save to our raw data folder
results_path = f"../data/raw/epl_results_{season_str}.csv"
df_results.to_csv(results_path, index=False)
print(f"Saved {len(df_results)} matches to {results_path}")

# Display the first few rows (including closing bookmaker odds)
df_results[['Date', 'HomeTeam', 'AwayTeam', 'FTHG', 'FTAG', 'B365H', 'B365D', 'B365A']].head()

Fetching match results from https://www.football-data.co.uk/mmz4281/2526/E0.csv...
Saved 380 matches to ../data/raw/epl_results_2526.csv


,Date,HomeTeam,AwayTeam,FTHG,FTAG,B365H,B365D,B365A
0,15/08/2025,Liverpool,Bournemouth,4,2,1.30,6.00,8.50
1,16/08/2025,Aston Villa,Newcastle,0,0,2.25,3.50,2.90
2,16/08/2025,Brighton,Fulham,1,1,1.91,3.60,4.00
3,16/08/2025,Sunderland,West Ham,3,0,3.25,3.40,2.25
4,16/08/2025,Tottenham,Burnley,3,0,1.38,4.75,8.50


In [2]:
from understatapi import UnderstatClient

print("Fetching advanced xG data from Understat...")

# The context manager automatically handles the connection session
with UnderstatClient() as understat:
    # Fetch all match data for the EPL season
    match_data = understat.league(league="EPL").get_match_data(season="2025")

# Convert the API's JSON response directly into a Pandas DataFrame
df_xg = pd.DataFrame(match_data)

# Save to our raw data folder
xg_path = f"../data/raw/understat_xg_{season_str}.csv"
df_xg.to_csv(xg_path, index=False)
print(f"Saved xG data to {xg_path}")

df_xg.head()

Fetching advanced xG data from Understat...
Saved xG data to ../data/raw/understat_xg_2526.csv


,id,isResult,h,a,goals,xG,datetime,forecast
0,28778,True,"{'id': '87', 'title': 'Liverpool', 'short_titl...","{'id': '73', 'title': 'Bournemouth', 'short_ti...","{'h': '4', 'a': '2'}","{'h': '2.33007', 'a': '1.57303'}",2025-08-15 19:00:00,"{'w': '0.5498', 'd': '0.2276', 'l': '0.2226'}"
1,28779,True,"{'id': '71', 'title': 'Aston Villa', 'short_ti...","{'id': '86', 'title': 'Newcastle United', 'sho...","{'h': '0', 'a': '0'}","{'h': '0.318601', 'a': '1.40098'}",2025-08-16 11:30:00,"{'w': '0.0629', 'd': '0.2371', 'l': '0.7'}"
2,28780,True,"{'id': '220', 'title': 'Brighton', 'short_titl...","{'id': '228', 'title': 'Fulham', 'short_title'...","{'h': '1', 'a': '1'}","{'h': '1.44008', 'a': '0.902883'}",2025-08-16 14:00:00,"{'w': '0.5158', 'd': '0.3088', 'l': '0.1754'}"
3,28781,True,"{'id': '77', 'title': 'Sunderland', 'short_tit...","{'id': '81', 'title': 'West Ham', 'short_title...","{'h': '3', 'a': '0'}","{'h': '0.724368', 'a': '0.558948'}",2025-08-16 14:00:00,"{'w': '0.363', 'd': '0.391', 'l': '0.246'}"
4,28782,True,"{'id': '82', 'title': 'Tottenham', 'short_titl...","{'id': '92', 'title': 'Burnley', 'short_title'...","{'h': '3', 'a': '0'}","{'h': '1.97858', 'a': '0.734138'}",2025-08-16 14:00:00,"{'w': '0.7032', 'd': '0.1993', 'l': '0.0975'}"


In [3]:
import numpy as np

# 1. Extract nested data from Understat
df_xg['HomeTeam_xG'] = df_xg['h'].apply(lambda x: x['title'])
df_xg['AwayTeam_xG'] = df_xg['a'].apply(lambda x: x['title'])
df_xg['Home_xG'] = df_xg['xG'].apply(lambda x: float(x['h']))
df_xg['Away_xG'] = df_xg['xG'].apply(lambda x: float(x['a']))

# Convert dates to match
df_xg['Date_xG'] = pd.to_datetime(df_xg['datetime']).dt.date
df_results['Date'] = pd.to_datetime(df_results['Date'], format='%d/%m/%Y').dt.date

# 2. Map Understat names to football-data names
name_mapping = {
    'Manchester United': 'Man United',
    'Manchester City': 'Man City',
    'Tottenham Hotspur': 'Tottenham',
    'Newcastle United': 'Newcastle',
    'Wolverhampton Wanderers': 'Wolves',
    'Nottingham Forest': 'Nott\'m Forest'
    # Note: Depending on the exact promoted teams for 25/26, 
    # you may need to add 1-2 more mappings here if the merge drops rows.
}

df_xg['HomeTeam_xG'] = df_xg['HomeTeam_xG'].replace(name_mapping)
df_xg['AwayTeam_xG'] = df_xg['AwayTeam_xG'].replace(name_mapping)

# 3. Merge the datasets on Date, HomeTeam, and AwayTeam
df_merged = pd.merge(
    df_results, 
    df_xg[['Date_xG', 'HomeTeam_xG', 'AwayTeam_xG', 'Home_xG', 'Away_xG']], 
    left_on=['Date', 'HomeTeam', 'AwayTeam'],
    right_on=['Date_xG', 'HomeTeam_xG', 'AwayTeam_xG'],
    how='inner'
)

print(f"Successfully merged {len(df_merged)} matches!")

Successfully merged 380 matches!


In [21]:
# Create a dataset from the perspective of the Home team
home_df = df_merged[['Date', 'HomeTeam', 'AwayTeam', 'FTHG', 'FTAG', 'Home_xG', 'Away_xG']].copy()
home_df.columns = ['Date', 'Team', 'Opponent', 'GoalsScored', 'GoalsConceded', 'xG_Created', 'xG_Conceded']
home_df['Venue'] = 'Home'

# Create a dataset from the perspective of the Away team
away_df = df_merged[['Date', 'AwayTeam', 'HomeTeam', 'FTAG', 'FTHG', 'Away_xG', 'Home_xG']].copy()
away_df.columns = ['Date', 'Team', 'Opponent', 'GoalsScored', 'GoalsConceded', 'xG_Created', 'xG_Conceded']
away_df['Venue'] = 'Away'

# Combine and sort chronologically
team_match_df = pd.concat([home_df, away_df]).sort_values(by=['Team', 'Date']).reset_index(drop=True)

In [22]:
# --- STEP 3: Calculate Rolling Averages safely using .transform() ---

# Ensure chronological ordering per team
team_match_df = team_match_df.sort_values(by=['Team', 'Date']).reset_index(drop=True)

# Calculate shifted 5-game rolling averages per team group
team_match_df['xG_Created_5g'] = team_match_df.groupby('Team')['xG_Created'].transform(
    lambda s: s.shift(1).rolling(window=5, min_periods=5).mean()
)
team_match_df['xG_Conceded_5g'] = team_match_df.groupby('Team')['xG_Conceded'].transform(
    lambda s: s.shift(1).rolling(window=5, min_periods=5).mean()
)
team_match_df['GoalsScored_5g'] = team_match_df.groupby('Team')['GoalsScored'].transform(
    lambda s: s.shift(1).rolling(window=5, min_periods=5).mean()
)
team_match_df['GoalsConceded_5g'] = team_match_df.groupby('Team')['GoalsConceded'].transform(
    lambda s: s.shift(1).rolling(window=5, min_periods=5).mean()
)

# Drop initial early-season rows with NaN rolling stats
team_match_df = team_match_df.dropna().reset_index(drop=True)

In [23]:
# --- STEP 4: Reassemble the ML Dataset ---

# Separate Home and Away subsets
home_stats = team_match_df[team_match_df['Venue'] == 'Home'].copy()
home_stats = home_stats.rename(columns={
    'xG_Created_5g': 'Home_xG_Created_5g',
    'xG_Conceded_5g': 'Home_xG_Conceded_5g'
})

away_stats = team_match_df[team_match_df['Venue'] == 'Away'].copy()
away_stats = away_stats.rename(columns={
    'xG_Created_5g': 'Away_xG_Created_5g',
    'xG_Conceded_5g': 'Away_xG_Conceded_5g'
})

# Construct base dataset
final_ml_df = df_merged[['Date', 'HomeTeam', 'AwayTeam', 'FTHG', 'FTAG', 'B365H', 'B365D', 'B365A']].copy()

# Merge Home stats
final_ml_df = pd.merge(
    final_ml_df, 
    home_stats[['Date', 'Team', 'Home_xG_Created_5g', 'Home_xG_Conceded_5g']], 
    left_on=['Date', 'HomeTeam'], 
    right_on=['Date', 'Team'], 
    how='inner'
).drop(columns=['Team'])

# Merge Away stats
final_ml_df = pd.merge(
    final_ml_df, 
    away_stats[['Date', 'Team', 'Away_xG_Created_5g', 'Away_xG_Conceded_5g']], 
    left_on=['Date', 'AwayTeam'], 
    right_on=['Date', 'Team'], 
    how='inner'
).drop(columns=['Team'])

# Save to CSV
processed_path = "../data/processed/epl_ml_features.csv"
final_ml_df.to_csv(processed_path, index=False)
print(f"Pipeline complete! {len(final_ml_df)} matches saved to {processed_path}")
final_ml_df.head()

Pipeline complete! 330 matches saved to ../data/processed/epl_ml_features.csv


,Date,HomeTeam,AwayTeam,FTHG,FTAG,B365H,B365D,B365A,Home_xG_Created_5g,Home_xG_Conceded_5g,Away_xG_Created_5g,Away_xG_Conceded_5g
0,2025-09-27,Brentford,Man United,3,1,3.40,3.80,2.00,1.180545,1.316786,1.961732,1.355401
1,2025-09-27,Chelsea,Brighton,1,3,1.85,3.80,3.80,1.795228,1.053770,1.552635,1.494381
2,2025-09-27,Crystal Palace,Liverpool,2,1,3.80,3.80,1.90,1.765370,0.944831,1.462923,0.982308
3,2025-09-27,Leeds,Bournemouth,2,2,3.10,3.25,2.38,0.900529,1.284295,1.494056,0.730532
4,2025-09-27,Man City,Burnley,5,1,1.17,7.50,15.00,1.760132,1.249190,0.874752,2.149584


In [26]:
import pandas as pd
import numpy as np
import os
from understatapi import UnderstatClient

# Your standard name mapping dictionary
name_mapping = {
    'Manchester United': 'Man United', 'Manchester City': 'Man City',
    'Tottenham Hotspur': 'Tottenham', 'Newcastle United': 'Newcastle',
    'Wolverhampton Wanderers': 'Wolves', 'Nottingham Forest': "Nott'm Forest",
    'West Ham United': 'West Ham', 'Brighton & Hove Albion': 'Brighton',
    'Leicester City': 'Leicester', 'Leeds United': 'Leeds',
    'Sheffield United': 'Sheffield Utd', 'Luton Town': 'Luton',
    'Ipswich Town': 'Ipswich', 'Southampton': 'Southampton',
    'Burnley': 'Burnley', 'Bournemouth': 'Bournemouth',
    'Aston Villa': 'Aston Villa', 'Everton': 'Everton',
    'Crystal Palace': 'Crystal Palace', 'Brentford': 'Brentford',
    'Fulham': 'Fulham', 'Norwich City': 'Norwich', 'Watford': 'Watford'
}

seasons_fd = ['2223', '2324', '2425', '2526']
seasons_us = ['2022', '2023', '2024', '2025']

multi_season_dfs = []

with UnderstatClient() as understat:
    for s_fd, s_us in zip(seasons_fd, seasons_us):
        print(f"Processing Season 20{s_fd[:2]}/20{s_fd[2:]} tabular data...")
        
        # 1. Download Match Context from football-data.co.uk
        url = f"https://www.football-data.co.uk/mmz4281/{s_fd}/E0.csv"
        df_res = pd.read_csv(url)
        df_res['Date'] = pd.to_datetime(df_res['Date'], format='%d/%m/%Y').dt.date
        
        # ADDED: Referee, Cards (Yellow/Red), Fouls, and Corners
        core_cols = ['Date', 'HomeTeam', 'AwayTeam', 'FTHG', 'FTAG', 
                     'Referee', 'HY', 'AY', 'HR', 'AR', 'HC', 'AC', 'HF', 'AF']
        odds_cols = ['B365H', 'B365D', 'B365A'] if 'B365H' in df_res.columns else []
        df_res = df_res[core_cols + odds_cols].dropna()

        # 2. Download Base xG Data from Understat
        match_data = understat.league(league="EPL").get_match_data(season=s_us)
        df_xg = pd.DataFrame(match_data)
        
        # ADDED: Save the specific match ID to fetch shot maps later
        df_xg['Understat_Match_ID'] = df_xg['id']
        df_xg['HomeTeam_xG'] = df_xg['h'].apply(lambda x: x['title']).replace(name_mapping)
        df_xg['AwayTeam_xG'] = df_xg['a'].apply(lambda x: x['title']).replace(name_mapping)
        df_xg['Home_xG'] = df_xg['xG'].apply(lambda x: float(x['h']))
        df_xg['Away_xG'] = df_xg['xG'].apply(lambda x: float(x['a']))
        df_xg['Date_xG'] = pd.to_datetime(df_xg['datetime']).dt.date

        # 3. Merge sources
        df_merged_season = pd.merge(
            df_res, 
            df_xg[['Understat_Match_ID', 'Date_xG', 'HomeTeam_xG', 'AwayTeam_xG', 'Home_xG', 'Away_xG']], 
            left_on=['Date', 'HomeTeam', 'AwayTeam'],
            right_on=['Date_xG', 'HomeTeam_xG', 'AwayTeam_xG'],
            how='inner'
        )
        
        df_merged_season['Season'] = s_us
        multi_season_dfs.append(df_merged_season)

df_multi_raw = pd.concat(multi_season_dfs, ignore_index=True).sort_values(by='Date').reset_index(drop=True)
os.makedirs('../data/raw', exist_ok=True)
df_multi_raw.to_csv('../data/raw/epl_multi_season_raw.csv', index=False)
print("Tabular data updated with Match Context!")

Processing Season 2022/2023 tabular data...
Processing Season 2023/2024 tabular data...
Processing Season 2024/2025 tabular data...
Processing Season 2025/2026 tabular data...
Tabular data updated with Match Context!


In [28]:
import json
import time
from understatapi import UnderstatClient

# Custom encoder to handle NumPy data types (like int64)
class NpEncoder(json.JSONEncoder):
    def default(self, obj):
        if isinstance(obj, np.integer):
            return int(obj)
        if isinstance(obj, np.floating):
            return float(obj)
        if isinstance(obj, np.ndarray):
            return obj.tolist()
        return super(NpEncoder, self).default(obj)

# Load the tabular data to get the IDs
df_raw = pd.read_csv('../data/raw/epl_multi_season_raw.csv')
match_ids = df_raw['Understat_Match_ID'].unique()

shot_data_list = []
print(f"Fetching shot maps for {len(match_ids)} matches. This will take ~10 minutes...")

with UnderstatClient() as understat:
    for i, match_id in enumerate(match_ids):
        try:
            # Request all shots for this specific match
            shots = understat.match(match=str(match_id)).get_shot_data()
            
            # Append our Match ID so we can link it back to the tabular data later
            shots['match_id'] = match_id
            shot_data_list.append(shots)
            
            # Polite rate limiting to avoid getting blocked
            time.sleep(0.4)
            
            if (i + 1) % 100 == 0:
                print(f"Processed {i + 1} / {len(match_ids)} matches...")
                
        except Exception as e:
            print(f"Failed to fetch match {match_id}: {e}")

# Save the complex JSON objects to a dedicated file
with open('../data/raw/epl_shot_maps.json', 'w') as f:
    json.dump(shot_data_list, f, cls=NpEncoder)

print("Shot maps successfully downloaded and saved to ../data/raw/epl_shot_maps.json!")

Fetching shot maps for 1482 matches. This will take ~10 minutes...
Processed 100 / 1482 matches...
Processed 200 / 1482 matches...
Processed 300 / 1482 matches...
Processed 400 / 1482 matches...
Processed 500 / 1482 matches...
Processed 600 / 1482 matches...
Processed 700 / 1482 matches...
Processed 800 / 1482 matches...
Processed 900 / 1482 matches...
Processed 1000 / 1482 matches...
Processed 1100 / 1482 matches...
Processed 1200 / 1482 matches...
Processed 1300 / 1482 matches...
Processed 1400 / 1482 matches...
Shot maps successfully downloaded and saved to ../data/raw/epl_shot_maps.json!


In [29]:
import pandas as pd
import numpy as np

# 1. Load the raw dataset with the new match context
df_raw = pd.read_csv('../data/raw/epl_multi_season_raw.csv')
df_raw['Date'] = pd.to_datetime(df_raw['Date'])

# 2. Restructure into a team-centric timeline (Stacking Home and Away)
# We pull in the Phase 2 context stats: Yellows (Y), Reds (R), Corners (C), Fouls (F)
home_cols = {
    'Date': 'Date', 'HomeTeam': 'Team', 'AwayTeam': 'Opponent', 
    'FTHG': 'GoalsScored', 'FTAG': 'GoalsConceded', 
    'Home_xG': 'xG_Created', 'Away_xG': 'xG_Conceded',
    'HY': 'YellowCards', 'HR': 'RedCards', 'HC': 'Corners', 'HF': 'Fouls'
}
home_df = df_raw[list(home_cols.keys())].rename(columns=home_cols)
home_df['Venue'] = 'Home'

away_cols = {
    'Date': 'Date', 'AwayTeam': 'Team', 'HomeTeam': 'Opponent', 
    'FTAG': 'GoalsScored', 'FTHG': 'GoalsConceded', 
    'Away_xG': 'xG_Created', 'Home_xG': 'xG_Conceded',
    'AY': 'YellowCards', 'AR': 'RedCards', 'AC': 'Corners', 'AF': 'Fouls'
}
away_df = df_raw[list(away_cols.keys())].rename(columns=away_cols)
away_df['Venue'] = 'Away'

# Combine and sort strictly chronologically per team
team_df = pd.concat([home_df, away_df]).sort_values(by=['Team', 'Date']).reset_index(drop=True)

# 3. Schedule Context (Rest Days & Congestion Flag)
team_df['Rest_Days'] = team_df.groupby('Team')['Date'].diff().dt.days
team_df['Rest_Days'] = team_df['Rest_Days'].fillna(14) # Default to 2 weeks for season openers/breaks
team_df['Congestion_Flag'] = (team_df['Rest_Days'] < 4).astype(int)

# 4. Multi-Scale Rolling Windows (Global Form)
# .shift(1) is critical here. It hides today's match stats from today's prediction line.
windows = [3, 5, 10]
features_to_roll = ['GoalsScored', 'GoalsConceded', 'xG_Created', 'xG_Conceded', 'Corners', 'Fouls']

for w in windows:
    for feat in features_to_roll:
        team_df[f'{feat}_roll{w}'] = team_df.groupby('Team')[feat].transform(
            lambda x: x.shift(1).rolling(window=w, min_periods=1).mean()
        )

# 5. Venue-Specific Form (e.g., Home performance only at Home)
# Grouping by both Team AND Venue to isolate away struggles or home fortresses
for feat in ['xG_Created', 'xG_Conceded']:
    team_df[f'{feat}_Venue_roll5'] = team_df.groupby(['Team', 'Venue'])[feat].transform(
        lambda x: x.shift(1).rolling(window=5, min_periods=1).mean()
    )

# 6. Re-merge into Match-Level Format for ML (Home vs Away)
# Add prefixes so we know which stats belong to which team
home_stats = team_df[team_df['Venue'] == 'Home'].add_prefix('Home_').rename(
    columns={'Home_Date': 'Date', 'Home_Team': 'HomeTeam'}
)
away_stats = team_df[team_df['Venue'] == 'Away'].add_prefix('Away_').rename(
    columns={'Away_Date': 'Date', 'Away_Team': 'AwayTeam'}
)

# Base match dataframe (keeping the referee and odds columns from Phase 1)
base_cols = ['Date', 'HomeTeam', 'AwayTeam', 'FTHG', 'FTAG', 'Referee']
odds_cols = [c for c in df_raw.columns if c in ['B365H', 'B365D', 'B365A']]
final_df = df_raw[base_cols + odds_cols].copy()

# Merge all engineered features back together
final_df = pd.merge(final_df, home_stats, on=['Date', 'HomeTeam'], how='inner')
final_df = pd.merge(final_df, away_stats, on=['Date', 'AwayTeam'], how='inner')

# Clean up redundant opponent/venue columns created during the merge
final_df = final_df.drop(columns=['Home_Opponent', 'Away_Opponent', 'Home_Venue', 'Away_Venue'])

# Drop early-season rows where rolling averages are just mirroring the first match
final_df = final_df.dropna().reset_index(drop=True)

# Save to processed folder
processed_path = '../data/processed/epl_model_features.csv'
final_df.to_csv(processed_path, index=False)
print(f"Phase 3 complete! Dataset ready with {len(final_df.columns)} features per match.")

Phase 3 complete! Dataset ready with 69 features per match.


In [30]:
import pandas as pd
import numpy as np

# 1. Load the raw dataset with match context
df_raw = pd.read_csv('../data/raw/epl_multi_season_raw.csv')
df_raw['Date'] = pd.to_datetime(df_raw['Date'])
df_raw = df_raw.sort_values('Date').reset_index(drop=True)

# 2. Calculate Elo Ratings Chronologically
def calculate_elo(df, k_factor=20, home_advantage=60):
    elo_dict = {}
    home_elos, away_elos = [], []

    for _, row in df.iterrows():
        home, away = row['HomeTeam'], row['AwayTeam']
        if home not in elo_dict: elo_dict[home] = 1500
        if away not in elo_dict: elo_dict[away] = 1500
        
        home_elo, away_elo = elo_dict[home], elo_dict[away]
        home_elos.append(home_elo)
        away_elos.append(away_elo)
        
        prob_home = 1 / (1 + 10 ** ((away_elo - (home_elo + home_advantage)) / 400))
        prob_away = 1 - prob_home
        
        if row['FTHG'] > row['FTAG']:
            res_home, res_away = 1, 0
        elif row['FTHG'] < row['FTAG']:
            res_home, res_away = 0, 1
        else:
            res_home, res_away = 0.5, 0.5
            
        elo_dict[home] = home_elo + k_factor * (res_home - prob_home)
        elo_dict[away] = away_elo + k_factor * (res_away - prob_away)
        
    df['Home_Elo'] = home_elos
    df['Away_Elo'] = away_elos
    return df

df_raw = calculate_elo(df_raw)

# 3. Restructure into team-centric timeline
home_cols = {
    'Date': 'Date', 'HomeTeam': 'Team', 'AwayTeam': 'Opponent', 
    'FTHG': 'GoalsScored', 'FTAG': 'GoalsConceded', 
    'Home_xG': 'xG_Created', 'Away_xG': 'xG_Conceded',
    'HY': 'YellowCards', 'HR': 'RedCards', 'HC': 'Corners', 'HF': 'Fouls'
}
home_df = df_raw[list(home_cols.keys())].rename(columns=home_cols)
home_df['Venue'] = 'Home'

away_cols = {
    'Date': 'Date', 'AwayTeam': 'Team', 'HomeTeam': 'Opponent', 
    'FTAG': 'GoalsScored', 'FTHG': 'GoalsConceded', 
    'Away_xG': 'xG_Created', 'Home_xG': 'xG_Conceded',
    'AY': 'YellowCards', 'AR': 'RedCards', 'AC': 'Corners', 'AF': 'Fouls'
}
away_df = df_raw[list(away_cols.keys())].rename(columns=away_cols)
away_df['Venue'] = 'Away'

team_df = pd.concat([home_df, away_df]).sort_values(by=['Team', 'Date']).reset_index(drop=True)

# 4. Schedule Context
team_df['Rest_Days'] = team_df.groupby('Team')['Date'].diff().dt.days
team_df['Rest_Days'] = team_df['Rest_Days'].fillna(14)
team_df['Congestion_Flag'] = (team_df['Rest_Days'] < 4).astype(int)

# 5. Multi-Scale Rolling Windows
windows = [3, 5, 10]
features_to_roll = ['GoalsScored', 'GoalsConceded', 'xG_Created', 'xG_Conceded', 'Corners', 'Fouls']

for w in windows:
    for feat in features_to_roll:
        team_df[f'{feat}_roll{w}'] = team_df.groupby('Team')[feat].transform(
            lambda x: x.shift(1).rolling(window=w, min_periods=1).mean()
        )

# 6. Venue-Specific Form
for feat in ['xG_Created', 'xG_Conceded']:
    team_df[f'{feat}_Venue_roll5'] = team_df.groupby(['Team', 'Venue'])[feat].transform(
        lambda x: x.shift(1).rolling(window=5, min_periods=1).mean()
    )

# 7. Re-merge back to match format, keeping Home_Elo and Away_Elo
home_stats = team_df[team_df['Venue'] == 'Home'].add_prefix('Home_').rename(
    columns={'Home_Date': 'Date', 'Home_Team': 'HomeTeam'}
)
away_stats = team_df[team_df['Venue'] == 'Away'].add_prefix('Away_').rename(
    columns={'Away_Date': 'Date', 'Away_Team': 'AwayTeam'}
)

base_cols = ['Date', 'HomeTeam', 'AwayTeam', 'FTHG', 'FTAG', 'Referee', 'Home_Elo', 'Away_Elo']
odds_cols = [c for c in df_raw.columns if c in ['B365H', 'B365D', 'B365A']]
final_df = df_raw[base_cols + odds_cols].copy()

final_df = pd.merge(final_df, home_stats, on=['Date', 'HomeTeam'], how='inner')
final_df = pd.merge(final_df, away_stats, on=['Date', 'AwayTeam'], how='inner')
final_df = final_df.drop(columns=['Home_Opponent', 'Away_Opponent', 'Home_Venue', 'Away_Venue'])
final_df = final_df.dropna().reset_index(drop=True)

# Save processed dataset
processed_path = '../data/processed/epl_model_features.csv'
final_df.to_csv(processed_path, index=False)
print(f"Phase 3 complete! Dataset updated with Elo ratings and saved to {processed_path}.")

Phase 3 complete! Dataset updated with Elo ratings and saved to ../data/processed/epl_model_features.csv.
